# Episode 1 — No `dac~`: Sound Is an Array

⏱ About 25 minutes · Difficulty ★☆☆

**Musical elements this episode controls: pitch, duration and volume.**

**The equivalent Max patch:** `cycle~ 110` → `*~ 0.3` → `dac~`, switched on for 2 seconds.

**What you will hear:** a clean, low hum (A2, 110 Hz) lasting 2 seconds. Later you will build a three-note melody and a chord yourself.

**By the end you can:**
- explain how a Max signal differs from a Python array
- write a sine from scratch, hear it and see it
- change pitch, duration and volume
- join notes into a melody and stack them into a chord
- save the sound as a `.wav` file

---

### The biggest difference between Max and Python

| | Max / RNBO | Python |
|---|---|---|
| Time | Runs by itself; turn on `dac~` and audio flows | There is no "running time" |
| Sound | Sent out one sample at a time, live | **The whole sound is computed at once** into an array |
| Listening | Immediate | Compute first, then play |

So building a synth in Python really means: **compute a long list of numbers, then play it.**

## 0. How to use this notebook

- Each grey box is a **cell**. Click into it and press **Shift + Enter** to run it.
- **Run from top to bottom, in order.** Later cells use what earlier cells computed.
- After changing a number in a cell, **re-run that cell and the cells below that depend on it** for the change to take effect. This differs from Max: there a change is live, here you "recompute".
- ⚠️ Turn your system volume down a little before you start.

The cell below "loads the tools":

| This line | What it is | Like, in Max |
|---|---|---|
| `numpy` (short: `np`) | math on whole lists of numbers | every math object with a `~` |
| `matplotlib` (short: `plt`) | plotting | `scope~` |
| `Audio` | a player | `dac~` |
| `wavfile` | read/write wav files | `sfrecord~` / write on `buffer~` |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from scipy.io import wavfile

## 1. What an array is

An array is **a list of numbers in a fixed order**. Start with one that holds just 4 numbers.

Four things to remember:

1. **Math happens on the whole thing.** `a * 2` multiplies every number by 2. No loop needed.
2. **`len(a)`** is how many numbers it holds.
3. **Square brackets pick a number, counting from 0.** `a[0]` is the first, `a[3]` the fourth.
4. **`a[:2]`** means "from the start up to, but not including, index 2" — the first two. This is slicing.

Run the cell and compare with the output.

In [ ]:
a = np.array([10, 20, 30, 40])

print("a        =", a)
print("a * 2    =", a * 2)
print("a + 1    =", a + 1)
print("len(a)   =", len(a))
print("a[0]     =", a[0])
print("a[3]     =", a[3])
print("a[:2]    =", a[:2])

**Compared with Max:** a signal patch cord carries endless samples and you never see their "numbers". An array is those samples **stopped, lined up and numbered**, so you can pick any one of them.

## 2. Three numbers

- `sr`: sample rate, samples per second (the number in Max's Audio Status)
- `freq`: frequency in Hz → sets the **pitch**
- `dur`: duration in seconds

In [ ]:
sr = 44100
freq = 110
dur = 2.0

## 3. The time table `t`

In Max, time runs by itself. In Python you first write down **at which second each sample happens**.

### Small numbers first

Suppose there are only 4 samples per second, for 2 seconds.

- `small_sr * small_dur` = 8 → 8 samples in total
- `np.arange(8)` → counts 0 to 7, the index of each sample
- `/ small_sr` → divide each index by 4; indices become seconds

In [ ]:
small_sr = 4
small_dur = 2

index = np.arange(int(small_sr * small_dur))   # the index of each sample: 0, 1, 2 ...
small_t = index / small_sr                     # index ÷ samples per second = which second

print("index   =", index)
print("small_t =", small_t)

Read the output: sample 0 is at 0 s, sample 1 at 0.25 s ... sample 4 lands exactly on 1 s (there are 4 per second).

What `int(...)` does: `4 * 2` here is really `8.0` (with a decimal point), but a count has to be a whole number, so `int` turns it into `8`.

### Real numbers

Exactly the same code, only with `sr = 44100`.

In [ ]:
# same as above: index ÷ sr = the time of each sample in seconds
t = np.arange(int(sr * dur)) / sr

print("number of samples:", len(t))
print("first five:", t[:5])
print("last one:", t[-1])

`t[-1]` means "the last one". It is not 2.0 but one sample short of 2 seconds — because we count from 0.

**`t` holds no sound yet, only time.**

## 4. The amplitude `x`

Now fill in an amplitude at each time point.

### Small numbers first

8 samples per second, frequency 1 Hz (1 cycle per second), 1 second only.

- `freq * t` → **how many cycles have elapsed** by that time point
- `2 * np.pi *` → `sin` wants an angle, and 1 cycle = 2π. Just a unit conversion.
- `np.sin(...)` → the height of the sine at that position

In [ ]:
tiny_t = np.arange(8) / 8                 # 1 second, 8 samples
cycles = 1 * tiny_t                       # freq × time = cycles elapsed
tiny_x = np.sin(2 * np.pi * cycles)       # cycles × 2π = angle, then take the sine

print("time           :", tiny_t)
print("cycles elapsed :", cycles)
print("amplitude      :", np.round(tiny_x, 2))

Read the `amplitude` row: 0 → 0.71 → 1 (the top) → 0.71 → 0 → -0.71 → -1 (the bottom) → -0.71. Exactly one full sine cycle.

(`np.round(..., 2)` only rounds to two decimals for printing.)

### Real numbers

In [ ]:
# freq * t = cycles elapsed; × 2π makes it an angle; sin gives the amplitude at that moment
x = np.sin(2 * np.pi * freq * t)

print("number of samples:", len(x))
print("t[100] =", t[100], "   x[100] =", x[100])

### About `t` and `x`

- both arrays have the same length (88200 numbers)
- **position is the index**: `t[100]` and `x[100]` describe the same sample; one is its time, the other its amplitude
- `x[100]` was computed from `t[100]`, so the order matches by itself; no extra labelling needed
- **`x` is the sound.** `t` exists only to compute `x` (and to label the plot axis)

## 5. Listen (this is your `dac~`)

`Audio(array, rate=sr)` shows a small player.

- first slot: the array to play
- `rate=sr`: samples per second. It needs this to know how fast to play
- `normalize=False`: do not auto-maximize the volume

**You should hear:** a clean, low hum for 2 seconds.

In [ ]:
Audio(0.3 * x, rate=sr, normalize=False)

### Volume

`0.3 * x` is `*~ 0.3`: every sample is multiplied by 0.3.

`x` originally sits between -1 and 1. That range is full scale; beyond it you clip.

**You should hear:** two tones below, the first clearly quieter than the second. Same pitch.

In [ ]:
display(Audio(0.05 * x, rate=sr, normalize=False))
display(Audio(0.4 * x, rate=sr, normalize=False))

(To show several players from one cell, wrap each in `display(...)`. With a single one you can leave it out.)

## 6. Look (this is your `scope~`)

`plt.plot(horizontal, vertical)`. Plot only the first 0.03 seconds; otherwise 88200 points are squeezed together and you see nothing.

`n = int(sr * 0.03)` is how many samples 0.03 seconds is; `t[:n]` and `x[:n]` take the first `n` of each.

**Guess before running:** how many cycles should a 110 Hz tone show in 0.03 seconds?

In [ ]:
n = int(sr * 0.03)                # how many samples 0.03 seconds is

plt.figure(figsize=(9, 3))
plt.plot(t[:n], x[:n])            # horizontal: time, vertical: amplitude, first n of each
plt.xlabel("time (s)")
plt.ylabel("amplitude")
plt.title(f"sine, {freq} Hz")
plt.show()

Answer: 110 × 0.03 = 3.3 cycles. Count the peaks in the plot.

### Zoom in until you see samples

Plot only the first 60 samples, with a dot on each. **The sound really is these dots; the line is only drawn between them.**

In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(t[:60], x[:60], marker="o", markersize=3)
plt.xlabel("time (s)")
plt.ylabel("amplitude")
plt.title("first 60 samples")
plt.show()

## 7. Wrapping things into your own objects

Typing `Audio(0.3 * x, rate=sr, normalize=False)` every time is tiring. In Max you would turn a group of objects you reuse into an **abstraction / subpatcher**. The Python equivalent is a **function** (`def`).

```python
def name(inlet1, inlet2):
    ...do things...
    return outlet
```

- what is in the parentheses are the **inlets** (called arguments)
- what follows `return` is the **outlet**
- `gain=0.3` gives that inlet a default: leave it out and it is 0.3
- the indented lines belong to the function

Three definitions follow; every later episode uses them:

- `play(x)` → `dac~` (the `np.clip` inside holds anything beyond ±1, protecting your ears)
- `scope(x)` → `scope~`
- `tone(freq, dur)` → `cycle~` switched on for a while

In [ ]:
def play(x, gain=0.3):
    # gain * x is the volume; np.clip holds anything beyond ±1
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # only plot this many samples from the start
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

def tone(freq, dur):
    t = np.arange(int(sr * dur)) / sr          # this t lives only inside the function and does not touch the outer t
    return np.sin(2 * np.pi * freq * t)

Defining a function **does nothing by itself** (like saving an abstraction without putting it in a patch). It runs when you "call" it:

**You should hear:** a 220 Hz sine for 1 second (an octave above the earlier one). The plot shows 6.6 cycles in 0.03 seconds.

In [ ]:
y = tone(220, 1.0)

scope(y, "tone(220, 1.0)")
play(y)

## 8. A melody: joining arrays

`np.concatenate([a, b, c])` joins several arrays **end to end** into one longer array.

In Max a melody needs `metro`, `counter`, `mtof` ... In Python, time is the position inside the array, so "after" simply means "appended".

**You should hear:** three notes in sequence, A – C# – E (a rising major arpeggio), 0.4 seconds each.

In [ ]:
melody = np.concatenate([tone(220.00, 0.4), tone(277.18, 0.4), tone(329.63, 0.4)])

print("total samples:", len(melody), "=", len(melody) / sr, "seconds")
play(melody)

You may hear a faint "tick" where one note meets the next. The earlier note did not end exactly on 0, so the waveform jumps. The envelope in Episode 5 fixes that.

## 9. A chord: adding arrays

**Joining = one after another. Adding = at the same time.**

`a + b` adds two arrays of equal length sample by sample. That is mixing — patching two cords into the same inlet in Max does exactly this.

Three sines added can reach an amplitude of 3, so lower the level (`gain=0.1`).

**You should hear:** the same three notes, now together: an A major chord for 2 seconds.

In [ ]:
chord = tone(220.00, 2.0) + tone(277.18, 2.0) + tone(329.63, 2.0)

print("max amplitude:", np.max(np.abs(chord)))
scope(chord, "three sines added")
play(chord, gain=0.1)

The waveform is no longer a sine — three sines stacked form a more complex shape. Keep that in mind: Episode 3 uses the same trick (stacking sines) to design timbres.

## 10. Saving a file

Python's output is not only the player. `wavfile.write(filename, sr, array)` writes a wav file next to this notebook, ready to drop into Max, Ableton or any DAW.

`.astype(np.float32)` stores the numbers as 32-bit floats (one of the standard wav formats).

In [ ]:
wavfile.write("ep01_melody.wav", sr, (0.3 * melody).astype(np.float32))
print("saved ep01_melody.wav")

## 11. A small experiment: lying about `rate`

An array is just numbers; it does not know "how many per second". The `rate` tells the player.

Suppose we lie and say the sample rate is half:

**Guess before running:** what happens to the pitch? To the duration?

In [ ]:
# sr // 2 = 22050: tell the player to play half as many samples per second
Audio(0.3 * x, rate=sr // 2, normalize=False)

Answer: an octave lower, and 4 seconds long. The same numbers played half as fast — the same thing as slowing a tape, or setting `groove~` to speed 0.5.

## Common mistakes

| Symptom | Cause |
|---|---|
| Changed `freq` but the sound is the same | The cell that computes `x` was not re-run |
| `NameError: name 'x' is not defined` | A cell above was skipped; run everything in order |
| The sound is harsh / blown out | Amplitude exceeds ±1; lower the gain |
| `operands could not be broadcast together` | Two arrays of different lengths were added |

## Exercises

**1.** Make and play a 440 Hz tone lasting 0.5 seconds.

<details><summary>Answer</summary>

```python
play(tone(440, 0.5))
```
</details>

**2.** Turn the melody of section 8 into a falling one (E – C# – A), 0.25 seconds per note. **You should hear:** three faster notes going down.

<details><summary>Answer</summary>

```python
play(np.concatenate([tone(329.63, 0.25), tone(277.18, 0.25), tone(220.00, 0.25)]))
```
</details>

**3.** Add `tone(220, 2.0)` and `tone(223, 2.0)` and play the result. **Guess first:** what will you hear?

<details><summary>Answer</summary>

```python
play(tone(220, 2.0) + tone(223, 2.0), gain=0.15)
```
One tone whose volume swells 3 times per second (beating). The frequencies differ by 3 Hz, so they cancel 3 times per second.
</details>

## Recap

| Max | Python |
|---|---|
| signal (always flowing) | array (computed all at once) |
| `cycle~ 110` | `np.sin(2 * np.pi * 110 * t)` |
| `*~ 0.3` | `0.3 * x` |
| two cords into one inlet | `a + b` |
| `dac~` | `play(x)` |
| `scope~` | `scope(x)` |
| abstraction | a `def` function |
| `sfrecord~` | `wavfile.write(...)` |

**Next episode:** `phasor~` — separating pitch from timbre.